# 🧠 FaceKey Animation Studio — T4 GPU Deep Learning Trainer
### Accelerated Speech-to-Facial Motion Training with Auto-Resume

**Features:**
- Uses Free **NVIDIA T4 16GB GPU** on Google Colab / Kaggle.
- Tensor Core Boost: **Automatic Mixed Precision (AMP FP16)** for maximum throughput.
- Unpacks dataset chunks from Google Drive directly to Colab high-speed local NVMe SSD.
- **Auto-Saves Checkpoints to Google Drive** (`checkpoints/checkpoint_latest.pt`).
- **Interruption-Proof**: If Colab disconnects, restarting this notebook immediately resumes training from the exact saved epoch and step!

## Step 1: Check GPU Acceleration & Tensor Cores
Verify that an NVIDIA T4 GPU is assigned to this Colab runtime.

In [ ]:
!nvidia-smi

## Step 2: Mount Google Drive
Connects your 5 TB Google Drive containing your dataset chunks and checkpoints.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
os.makedirs(f'{DRIVE_DIR}/chunks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'[✓] Google Drive mounted: {DRIVE_DIR}')

## Step 2b: Auto-Sync All Multi-Cloud Chunks into 5 TB Google Drive
Pulls all chunks collected across Kaggle, Lightning.ai, and Local PC directly into `/content/drive/MyDrive/FaceKeyDataset/chunks/`.
- **Fast Datacenter Transfer**: ~100 MB/s cloud-to-cloud.
- **Zero Duplicates**: Skips files that already exist on Drive.
- **0 MB Local PC Internet**: Uses Google Colab's cloud bandwidth.

In [ ]:
# ==============================================================================
# 🔄 AUTO-SYNC ALL MULTI-CLOUD CHUNKS INTO 5 TB GOOGLE DRIVE
# ==============================================================================
import os
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
token_file = '/content/drive/MyDrive/FaceKeyDataset/hf_token.txt'
if not HF_TOKEN and os.path.exists(token_file):
    with open(token_file) as f:
        HF_TOKEN = f.read().strip()

if not HF_TOKEN:
    HF_TOKEN = input('Enter Hugging Face Token: ').strip()
    if HF_TOKEN and os.path.exists('/content/drive/MyDrive/FaceKeyDataset'):
        with open(token_file, 'w') as f:
            f.write(HF_TOKEN)

try:
    print('[*] Syncing multi-cloud chunks (Kaggle + Lightning + Local PC) into 5 TB Drive...')
    snapshot_download(
        repo_id=HF_REPO,
        repo_type='dataset',
        token=HF_TOKEN,
        local_dir='/content/drive/MyDrive/FaceKeyDataset'
    )
    print('[+] Google Drive is 100% up-to-date with all multi-cloud chunks!')
except Exception as e:
    print(f'[!] Sync Note: {e}')


## Step 3: Clone Repository & Install Training Requirements (Self-Healing)

In [ ]:
import os, sys, shutil

# Always navigate to root /content first so no working directory locking errors occur
%cd /content

REPO_URL = 'https://github.com/Vijay-1010110/FaceKeyAnimation.git'
APP_DIR = '/content/FaceKeyAnimation'

# Self-healing clone / reset to ensure 100% clean, latest code from GitHub
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating existing repository to latest origin/main...')
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print('[*] Performing fresh clone of FaceKeyAnimation...')
    !rm -rf {APP_DIR}
    !git clone {REPO_URL} {APP_DIR}

%cd {APP_DIR}
!pip install -q torch torchvision torchaudio numpy scipy
print('[+] Training environment and repository ready!')

## Step 4: Launch Deep Learning Training (Auto-Resumes from Checkpoint)
This script:
1. Unpacks dataset `.tar.gz` chunks from Drive into `/content/training_data/` local SSD.
2. Trains the `SpeechToFaceAnimator` model with FP16 Tensor Core acceleration.
3. Checks if `checkpoint_latest.pt` exists in Google Drive to resume automatically.
4. Saves best and latest weights directly back to Google Drive after each epoch.

In [ ]:
!python scripts/train_speech_to_animation.py \
    --drive-dir "/content/drive/MyDrive/FaceKeyDataset" \
    --epochs 50 \
    --batch-size 64 \
    --lr 0.0001 \
    --seq-len 64